In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt


mat = scipy.io.loadmat(r'C:\Users\ahmtb\Desktop\5. Battery Data Set\1. BatteryAgingARC-FY08Q4\B0005.mat')

print(mat.keys())

In [ ]:
import pprint
pprint.pprint(mat['B0005'].dtype)

In [ ]:
cycles = mat['B0005'][0][0]['cycle'][0]
print('Toplam döngü sayısı:', len(cycles))
print('İlk döngü tipi:', cycles[0]['type'][0])

In [ ]:
print('İlk döngünün içindekiler:', cycles[0].dtype)

In [ ]:
print('Data içindekiler:', cycles[0]['data'][0][0].dtype)

In [ ]:

data_charge = cycles[0]['data'][0][0]

voltage_c = data_charge['Voltage_measured'][0].flatten()
current_c = data_charge['Current_measured'][0].flatten()
temperature_c = data_charge['Temperature_measured'][0].flatten()
time_c = data_charge['Time'][0].flatten()

print('Charge veri sayısı:', len(voltage_c))

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(time_c, voltage_c, color='blue')
axes[0].set_title('Voltaj (V) - Charge')
axes[0].set_ylabel('Volt')

axes[1].plot(time_c, current_c, color='red')
axes[1].set_title('Akım (A) - Charge')
axes[1].set_ylabel('Amper')

axes[2].plot(time_c, temperature_c, color='green')
axes[2].set_title('Sıcaklık (°C) - Charge')
axes[2].set_ylabel('Celsius')
axes[2].set_xlabel('Zaman (s)')

plt.tight_layout()
plt.show()

In [ ]:
data = cycles[1]['data'][0][0]

voltage = data['Voltage_measured'][0].flatten()
current = data['Current_measured'][0].flatten()
temperature = data['Temperature_measured'][0].flatten()
time = data['Time'][0].flatten()

print('Veri sayısı:', len(voltage))

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(time, voltage, color='blue')
axes[0].set_title('Voltaj (V)')
axes[0].set_ylabel('Volt')

axes[1].plot(time, current, color='red')
axes[1].set_title('Akım (A)')
axes[1].set_ylabel('Amper')

axes[2].plot(time, temperature, color='green')
axes[2].set_title('Sıcaklık (°C)')
axes[2].set_ylabel('Celsius')
axes[2].set_xlabel('Zaman (s)')

plt.tight_layout()
plt.show()

In [ ]:
capacities = []
cycle_numbers = []

for i, cycle in enumerate(cycles):
    if cycle['type'][0] == 'discharge':
        data = cycle['data'][0][0]
        current = data['Current_measured'][0].flatten()
        time = data['Time'][0].flatten()
        
        
        capacity = abs(np.trapezoid(current, time)) / 3600
        capacities.append(capacity)
        cycle_numbers.append(i)

capacities = np.array(capacities)
cycle_numbers = np.array(cycle_numbers)


soh = (capacities / capacities[0]) * 100

print(f'Toplam discharge döngüsü: {len(capacities)}')
print(f'İlk kapasite: {capacities[0]:.4f} Ah')
print(f'Son kapasite: {capacities[-1]:.4f} Ah')
print(f'Son SOH: %{soh[-1]:.1f}')


plt.figure(figsize=(12, 5))
plt.plot(range(len(soh)), soh, color='purple', linewidth=2)
plt.axhline(y=80, color='red', linestyle='--', label='EOL sınırı (%80)')
plt.title('Batarya SOH Degradasyon Eğrisi')
plt.xlabel('Döngü Sayısı')
plt.ylabel('SOH (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

battery_files = {
    'B0005': r'C:\Users\ahmtb\Desktop\5. Battery Data Set\1. BatteryAgingARC-FY08Q4\B0005.mat',
    'B0006': r'C:\Users\ahmtb\Desktop\5. Battery Data Set\1. BatteryAgingARC-FY08Q4\B0006.mat',
    'B0007': r'C:\Users\ahmtb\Desktop\5. Battery Data Set\1. BatteryAgingARC-FY08Q4\B0007.mat',
    'B0018': r'C:\Users\ahmtb\Desktop\5. Battery Data Set\1. BatteryAgingARC-FY08Q4\B0018.mat',
}

batteries = {}
for name, path in battery_files.items():
    mat = scipy.io.loadmat(path)
    batteries[name] = mat[name][0][0]['cycle'][0]
    print(f'{name} yüklendi — {len(batteries[name])} döngü')

In [ ]:
from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()
soh_normalized = scaler.fit_transform(soh.reshape(-1, 1)).flatten()


def create_sequences(data, seq_length=10):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10  

X, y = create_sequences(soh_normalized, SEQ_LENGTH)


split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


X_train = X_train.reshape(-1, SEQ_LENGTH, 1)
X_test = X_test.reshape(-1, SEQ_LENGTH, 1)

print(f'Train boyutu: {X_train.shape}')
print(f'Test boyutu: {X_test.shape}')
print('Veri hazır! ✅')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

model = Sequential([
    Input(shape=(SEQ_LENGTH, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

In [ ]:

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)


plt.figure(figsize=(12, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Eğitim Süreci')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

y_pred = model.predict(X_test)


y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_real = scaler.inverse_transform(y_pred).flatten()


mae = np.mean(np.abs(y_test_real - y_pred_real))
rmse = np.sqrt(np.mean((y_test_real - y_pred_real)**2))

print(f'MAE:  {mae:.4f}%')
print(f'RMSE: {rmse:.4f}%')


plt.figure(figsize=(12, 5))
plt.plot(y_test_real, label='Gerçek SOH', color='blue', linewidth=2)
plt.plot(y_pred_real, label='Tahmin SOH', color='red', linestyle='--', linewidth=2)
plt.axhline(y=80, color='green', linestyle='--', label='EOL sınırı (%80)')
plt.title('Gerçek vs Tahmin SOH')
plt.xlabel('Döngü')
plt.ylabel('SOH (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

model.save('batarya_soh_model.keras')


import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model kaydedildi! ✅')
print('Scaler kaydedildi! ✅')

In [ ]:
!C:\Users\ahmtb\AppData\Local\Programs\Python\Python311\python.exe -m pip install fastapi uvicorn

In [ ]:
def get_soh(cycles):
    capacities = []
    for cycle in cycles:
        if cycle['type'][0] == 'discharge':
            data = cycle['data'][0][0]
            current = data['Current_measured'][0].flatten()
            time = data['Time'][0].flatten()
            capacity = abs(np.trapezoid(current, time)) / 3600
            capacities.append(capacity)
    capacities = np.array(capacities)
    soh = (capacities / capacities[0]) * 100
    return soh


soh_dict = {}
for name, cycles in batteries.items():
    soh_dict[name] = get_soh(cycles)
    print(f'{name} — {len(soh_dict[name])} discharge döngüsü — Son SOH: %{soh_dict[name][-1]:.1f}')


plt.figure(figsize=(14, 6))
colors = {'B0005': 'purple', 'B0006': 'blue', 'B0007': 'green', 'B0018': 'orange'}

for name, soh in soh_dict.items():
    plt.plot(range(len(soh)), soh, label=name, color=colors[name], linewidth=2)

plt.axhline(y=80, color='red', linestyle='--', label='EOL sınırı (%80)')
plt.title('Tüm Bataryalar — SOH Degradasyon Karşılaştırması')
plt.xlabel('Döngü Sayısı')
plt.ylabel('SOH (%)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

all_soh = np.concatenate([soh_dict['B0005'], soh_dict['B0006'], 
                          soh_dict['B0007'], soh_dict['B0018']])

print(f'Toplam veri noktası: {len(all_soh)}')


scaler_all = MinMaxScaler()
all_soh_normalized = scaler_all.fit_transform(all_soh.reshape(-1, 1)).flatten()


X_all, y_all = create_sequences(all_soh_normalized, SEQ_LENGTH)


split = int(len(X_all) * 0.8)
X_train_all, X_test_all = X_all[:split], X_all[split:]
y_train_all, y_test_all = y_all[:split], y_all[split:]

X_train_all = X_train_all.reshape(-1, SEQ_LENGTH, 1)
X_test_all = X_test_all.reshape(-1, SEQ_LENGTH, 1)

print(f'Train: {X_train_all.shape}')
print(f'Test: {X_test_all.shape}')

In [ ]:
model_all = Sequential([
    Input(shape=(SEQ_LENGTH, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model_all.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_all = model_all.fit(
    X_train_all, y_train_all,
    epochs=100,
    batch_size=16,
    validation_data=(X_test_all, y_test_all),
    verbose=1
)

plt.figure(figsize=(12, 4))
plt.plot(history_all.history['loss'], label='Train Loss')
plt.plot(history_all.history['val_loss'], label='Val Loss')
plt.title('Çoklu Batarya Model Eğitimi')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

y_pred_all = model_all.predict(X_test_all)


y_test_real_all = scaler_all.inverse_transform(y_test_all.reshape(-1, 1)).flatten()
y_pred_real_all = scaler_all.inverse_transform(y_pred_all).flatten()


mae_all = np.mean(np.abs(y_test_real_all - y_pred_real_all))
rmse_all = np.sqrt(np.mean((y_test_real_all - y_pred_real_all)**2))

print(f'Çoklu Batarya MAE:  {mae_all:.4f}%')
print(f'Çoklu Batarya RMSE: {rmse_all:.4f}%')
print(f'Önceki MAE:  0.5823%')
print(f'Önceki RMSE: 0.7512%')


plt.figure(figsize=(12, 5))
plt.plot(y_test_real_all, label='Gerçek SOH', color='blue', linewidth=2)
plt.plot(y_pred_real_all, label='Tahmin SOH', color='red', linestyle='--', linewidth=2)
plt.axhline(y=80, color='green', linestyle='--', label='EOL sınırı (%80)')
plt.title('Çoklu Batarya — Gerçek vs Tahmin SOH')
plt.xlabel('Döngü')
plt.ylabel('SOH (%)')
plt.legend()
plt.grid(True)
plt.show()


model_all.save('batarya_soh_model.keras')
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler_all, f)

print('Yeni model kaydedildi! ✅')

In [ ]:

def get_rul(soh, eol_threshold=80):
    rul = []
    for i in range(len(soh)):
        
        remaining = np.where(soh[i:] < eol_threshold)[0]
        if len(remaining) > 0:
            rul.append(remaining[0])
        else:
            rul.append(len(soh) - i)  
    return np.array(rul)


plt.figure(figsize=(14, 6))
colors = {'B0005': 'purple', 'B0006': 'blue', 'B0007': 'green', 'B0018': 'orange'}

for name, soh in soh_dict.items():
    rul = get_rul(soh)
    plt.plot(range(len(rul)), rul, label=name, color=colors[name], linewidth=2)
    print(f'{name} — Başlangıç RUL: {rul[0]} döngü')

plt.title('Tüm Bataryalar — RUL (Kalan Kullanılabilir Ömür)')
plt.xlabel('Döngü Sayısı')
plt.ylabel('Kalan Döngü')
plt.legend()
plt.grid(True)
plt.show()